# Stage 1a — Misconception Taxonomy Generation

**AAI-590 Capstone · Monish Yarapathineni**

Derives a 6–8 category misconception taxonomy by clustering the expert-authored
misconception descriptions from the Eedi *Mining Misconceptions in Mathematics*
corpus.

**Why an external corpus.** The taxonomy is derived from Eedi rather than from
FoundationalASSIST wrong answers for two reasons. First, Eedi entries are
natural-language *descriptions* of misconceptions authored and validated by
teachers, whereas FoundationalASSIST wrong answers are bare response strings
(`12`, `3/4`) that carry little standalone semantic signal. Second, deriving
categories from the same data they will later label introduces circularity —
an independently sourced taxonomy avoids fitting the categories to the target set.

**Pipeline position.** Stage 1a (this notebook) produces the taxonomy.
Stage 1b applies it to 20,520 unique (problem, wrong answer) pairs from
FoundationalASSIST. Stage 2 trains the LSTM on the resulting labels.

| Section | Purpose |
|---|---|
| 1 | Install dependencies |
| 2 | Download Eedi corpus via kagglehub |
| 3 | Load and inspect misconception descriptions |
| 4 | Configure Anthropic client |
| 5 | Build clustering prompt |
| 6 | Generate taxonomy |
| 7 | Stability check across repeated runs |
| 8 | Coverage validation against the full corpus |
| 9 | Save taxonomy artifact |

---

## 1 · Install dependencies

In [1]:
%pip install -q kagglehub anthropic

import json, re, os, time, textwrap
from collections import Counter

import kagglehub
import pandas as pd

print("kagglehub", kagglehub.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 6.9 MB/s eta 0:00:00
kagglehub 1.0.2


## 2 · Download the Eedi corpus

`competition_download` requires Kaggle credentials **and** prior acceptance of the
competition rules on the website. If you have not already, visit the competition
page and click *Late Submission* / *Join Competition* to unlock the data:

https://www.kaggle.com/competitions/eedi-mining-misconceptions-in-mathematics/data

Credentials can be supplied either by adding `KAGGLE_USERNAME` / `KAGGLE_KEY` to
Colab secrets (left sidebar → 🔑) or interactively via `kagglehub.login()`.

In [11]:
# Optional: pull Kaggle credentials from Colab secrets if present.
try:
    from google.colab import userdata
    for k in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
        try:
            os.environ[k] = userdata.get(k)
        except Exception:
            pass
except ImportError:
    pass

if not os.environ.get("KAGGLE_KEYm"):
    kagglehub.login()   # interactive fallback

path = kagglehub.competition_download('eedi-mining-misconceptions-in-mathematics')

print("Path to competition files:", path)
print()
for f in sorted(os.listdir(path)):
    size_mb = os.path.getsize(os.path.join(path, f)) / 1e6
    print(f"  {f:<32} {size_mb:>8.2f} MB")

100%|██████████| 260k/260k [00:00<00:00, 59.5MB/s]

Extracting files...
Path to competition files: /root/.cache/kagglehub/competitions/eedi-mining-misconceptions-in-mathematics

  misconception_mapping.csv            0.20 MB
  sample_submission.csv                0.00 MB
  test.csv                             0.00 MB
  train.csv                            0.71 MB


## 3 · Load and inspect the misconception descriptions

`misconception_mapping.csv` is the only file Stage 1a needs. It maps each
misconception id to its natural-language description.

In [12]:
mis = pd.read_csv(os.path.join(path, "misconception_mapping.csv"))

print("Shape:", mis.shape)
print("Columns:", list(mis.columns))
print()
display(mis.head(10))

# Locate the free-text column regardless of exact header casing.
text_col = next(col for col in mis.columns if "name" in col.lower() or "misconception" in col.lower() and mis[col].dtype == object)
print("Using text column:", text_col)

Shape: (2587, 2)
Columns: ['MisconceptionId', 'MisconceptionName']



,MisconceptionId,MisconceptionName
0,0,Does not know that angles in a triangle sum to...
1,1,Uses dividing fractions method for multiplying...
2,2,Believes there are 100 degrees in a full turn
3,3,Thinks a quadratic without a non variable term...
4,4,Believes addition of terms and powers of terms...
5,5,"When measuring a reflex angle, gives the acute..."
6,6,Can identify the multiplier used to form an eq...
7,7,Believes gradient = change in y
8,8,Student thinks that any two angles along a str...
9,9,Thinks there are 180 degrees in a full turn


Using text column: MisconceptionName


In [13]:
descriptions = (
    mis[text_col]
    .dropna()
    .astype(str)
    .str.strip()
)
descriptions = descriptions[descriptions.str.len() > 0].tolist()

print(f"Total misconception descriptions : {len(descriptions):,}")
print(f"Unique                            : {len(set(descriptions)):,}")
print()

lengths = pd.Series([len(d) for d in descriptions])
print("Description length (characters)")
print(lengths.describe().round(1).to_string())
print()

print("Sample of 15:")
for d in descriptions[:15]:
    print("  •", textwrap.shorten(d, 110))

Total misconception descriptions : 2,587
Unique                            : 2,586

Description length (characters)
count    2587.0
mean       70.4
std        29.4
min        14.0
25%        49.0
50%        66.0
75%        86.0
max       208.0

Sample of 15:
  • Does not know that angles in a triangle sum to 180 degrees
  • Uses dividing fractions method for multiplying fractions
  • Believes there are 100 degrees in a full turn
  • Thinks a quadratic without a non variable term, can not be factorised
  • Believes addition of terms and powers of terms are equivalent e.g. a + c = a^c
  • When measuring a reflex angle, gives the acute or obtuse angle that sums to 360 instead
  • Can identify the multiplier used to form an equivalent fraction but does not apply to the numerator
  • Believes gradient = change in y
  • Student thinks that any two angles along a straight line are equal
  • Thinks there are 180 degrees in a full turn
  • Believes duration can be read from a timetable, rather 

In [14]:
# Rough token estimate for the single clustering call (~4 chars per token).
corpus = "\n".join(f"{i+1}. {d}" for i, d in enumerate(descriptions))
est_tokens = len(corpus) / 4

print(f"Corpus characters      : {len(corpus):,}")
print(f"Estimated input tokens : {est_tokens:,.0f}")
print()
print("Comfortably within a single context window."
      if est_tokens < 150_000 else
      "WARNING: large input — consider chunked clustering.")

Corpus characters      : 199,063
Estimated input tokens : 49,766

Comfortably within a single context window.


## 4 · Configure the Anthropic client

Add `ANTHROPIC_API_KEY` to Colab secrets (left sidebar → 🔑) and enable notebook
access. The key is read into the runtime only — never hardcode it in a cell, and
never commit it.

**Model choice.** Stage 1a is a single call producing the artifact every
downstream stage depends on, so it uses Opus. Stage 1b, which processes 20,520
pairs, uses Sonnet for cost reasons.

In [15]:
from anthropic import Anthropic

try:
    from google.colab import userdata
    api_key = userdata.get("ANTHROPIC_API_KEY")
except ImportError:
    api_key = os.environ.get("ANTHROPIC_API_KEY")

assert api_key, "ANTHROPIC_API_KEY not found — add it to Colab secrets."

client = Anthropic(api_key=api_key)

TAXONOMY_MODEL = "claude-opus-5"   # Stage 1a — one-time, quality-critical
LABELING_MODEL = "claude-sonnet-5" # Stage 1b — high volume (used in notebook 03)

print("Client ready. Taxonomy model:", TAXONOMY_MODEL)

Client ready. Taxonomy model: claude-opus-5


## 5 · Build the clustering prompt

Three constraints do the real work here:

1. **Orthogonal to topic.** Left unconstrained, an LLM clusters these into
   *Fractions / Algebra / Geometry*. That is useless for this project — the
   knowledge component field already encodes topic. The categories must describe
   the **nature of the error**, and must apply across every mathematical domain.
2. **Mutually exclusive.** The output is a single-label classification target for
   the LSTM, so overlapping categories degrade the training signal.
3. **Pedagogically actionable.** Each category should imply a different teaching
   response, which is the entire justification for predicting misconception
   *type* rather than plain correctness.

In [16]:
SYSTEM_PROMPT = """You are an expert in mathematics education and learning \
science, assisting with the design of a misconception taxonomy for an \
educational machine learning system."""

USER_PROMPT_TEMPLATE = """Below are {n} misconception descriptions authored by \
mathematics teachers. Each describes a specific incorrect belief or faulty \
procedure a student may hold.

Your task is to cluster these into 6-8 categories that will serve as the target \
classes for a machine learning classifier.

CRITICAL CONSTRAINTS:

1. Categories must describe the NATURE OF THE ERROR, not the mathematical topic.
   Do NOT create categories like "Fractions", "Algebra", or "Geometry". The
   system already tracks topic separately via knowledge components. A single
   category must be applicable across arithmetic, algebra, geometry, and
   statistics alike.

2. Categories must be mutually exclusive. Each will be used as a single-label
   classification target, so a given misconception should fall clearly into
   exactly one category.

3. Categories must be pedagogically actionable. Each should imply a materially
   different instructional response from a teacher. If two categories would lead
   a teacher to do the same thing, merge them.

4. Categories must collectively cover the full corpus. Avoid a residual
   "Other" or "Miscellaneous" bucket.

5. Choose the number of categories (between 6 and 8) that best fits the natural
   structure of the data. Justify the number you chose.

Return ONLY a JSON object in this exact schema, with no prose before or after:

{{
  "n_categories": <int>,
  "rationale_for_count": "<why this number rather than more or fewer>",
  "categories": [
    {{
      "name": "<short label, 2-4 words>",
      "definition": "<1-2 sentences a teacher would understand>",
      "distinguishing_test": "<the question that separates this from adjacent categories>",
      "instructional_response": "<what a teacher should do differently for this>",
      "example_misconceptions": ["<verbatim from the list>", "<verbatim>", "<verbatim>"],
      "estimated_share": <float 0-1, approximate fraction of corpus>
    }}
  ]
}}

MISCONCEPTION DESCRIPTIONS:

{corpus}"""

user_prompt = USER_PROMPT_TEMPLATE.format(n=len(descriptions), corpus=corpus)
print(f"Prompt built — {len(user_prompt):,} characters")

Prompt built — 201,022 characters


## 6 · Generate the taxonomy

In [22]:
def extract_json(text):
    """Pull a JSON object out of a model response, tolerating code fences and attempting common repairs."""

    raw_content_to_parse = None

    # Try to find fenced JSON first
    fenced_match = re.search(r"```(?:json)?\s*(.*?)\s*```", text, re.S)
    if fenced_match:
        raw_content_to_parse = fenced_match.group(1)
        try:
            return json.loads(raw_content_to_parse)
        except json.JSONDecodeError as e:
            # If fenced JSON is malformed, print a warning and fall back to unfenced or repaired logic
            print(f"Warning: Failed to parse JSON inside code fence. Error: {e}. Attempting unfenced or repaired extraction.")
            raw_content_to_parse = None # Reset to try other methods

    # If no fenced block or fenced parsing failed, try to extract between outermost curly braces
    if raw_content_to_parse is None:
        try:
            start = text.index("{")
            end = text.rindex("}") + 1
            raw_content_to_parse = text[start:end]
        except ValueError: # No curly braces found, or JSON was truncated
            raise ValueError(f"Could not find a JSON object in the model response. Full text:\n{text}")

    # Attempt common repairs before final json.loads
    if raw_content_to_parse:
        # Repair missing commas between objects in an array: `}{` -> `},{`
        repaired_content = re.sub(r'}\s*{', '},{', raw_content_to_parse)

        try:
            return json.loads(repaired_content)
        except json.JSONDecodeError as e:
            # If all attempts fail, raise a clear error, showing the repaired text if available
            error_message = f"Could not extract a valid JSON object from the model response. Error: {e}."
            if raw_content_to_parse != repaired_content:
                error_message += f"\nAttempted repair. Original text (partial):\n{raw_content_to_parse[:500]}...\nRepaired text (partial):\n{repaired_content[:500]}..."
            else:
                error_message += f"\nFull text (partial):\n{raw_content_to_parse[:500]}..."
            raise ValueError(error_message) from e


def generate_taxonomy(temperature=1.0):
    resp = client.messages.create(
        model=TAXONOMY_MODEL,
        max_tokens=16000, # Increased max_tokens to prevent truncation
        temperature=temperature,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": user_prompt}],
    )
    usage = resp.usage

    response_text = ""
    for block in resp.content:
        if block.type == "text":
            response_text += block.text
            break

    if not response_text:
        raise ValueError("No text content of type 'text' found in the model response.")

    return extract_json(response_text), usage


taxonomy, usage = generate_taxonomy()

print(f"Input tokens  : {usage.input_tokens:,}")
print(f"Output tokens : {usage.output_tokens:,}")
print(f"Categories    : {taxonomy['n_categories']}")
print()
print("Rationale:", taxonomy["rationale_for_count"])

Input tokens  : 65,351
Output tokens : 8,501
Categories    : 8

Rationale: Sorting the corpus by 'what went wrong' rather than 'what topic' reveals eight recurring failure modes that persist across arithmetic, algebra, geometry and statistics: (1) the language/symbol layer, (2) the stored-fact layer, (3) execution of a known algorithm, (4) illegitimate transfer of a rule to a context where it does not hold, (5) the meaning of the underlying object or relationship, (6) extracting information from a representation, (7) spatial visualisation and appearance-based inference, and (8) reading the task itself and monitoring what was asked. Collapsing to six forces two damaging merges: terminology-vs-fact recall (vocabulary discrimination work vs. retrieval/derivation practice) and representation-reading-vs-spatial-visualisation (careful data extraction routines vs. manipulatives and dynamic geometry) — each pair demands genuinely different lessons. Going beyond eight would split categories tha

In [23]:
for i, cat in enumerate(taxonomy["categories"], 1):
    print(f"\n{'='*72}")
    print(f"{i}. {cat['name'].upper()}   (~{cat.get('estimated_share', 0):.0%} of corpus)")
    print("=" * 72)
    print(f"\nDefinition\n  {textwrap.fill(cat['definition'], 68, subsequent_indent='  ')}")
    print(f"\nDistinguishing test\n  {textwrap.fill(cat['distinguishing_test'], 68, subsequent_indent='  ')}")
    print(f"\nInstructional response\n  {textwrap.fill(cat['instructional_response'], 68, subsequent_indent='  ')}")
    print("\nExamples")
    for ex in cat["example_misconceptions"][:3]:
        print(f"  • {textwrap.shorten(ex, 66)}")


1. TERMINOLOGY & NOTATION CONFUSION   (~11% of corpus)

Definition
  The student's error is located in mathematical language, labels,
  symbols or conventions: two named objects are swapped, a term's
  meaning is unknown, or a notational convention is misread or
  misused. The underlying arithmetic or reasoning may be fine once
  the label is fixed.

Distinguishing test
  If you told the student what the word, letter, label or symbol
  means, would the error disappear? If yes, it is
  terminology/notation; if they would still misremember a numerical
  rule or property, it is Faulty Fact or Formula Recall.

Instructional response
  Explicit vocabulary and notation instruction: side-by-side
  examples/non-examples of the confused pair, concept cards, always-
  sometimes-never sorting of definitions, and insisting students
  read notation aloud correctly before computing.

Examples
  • Confuses the terms edges and vertices
  • Does not know the meaning of the word parallel
  • Confuses a

## 7 · Stability check

A single clustering pass could be an artifact of sampling. Running it twice more
and comparing category counts and names indicates whether the structure is
genuinely present in the corpus or incidental.

This is cheap — three calls total — and the result is worth reporting in the
Methodology section.

In [24]:
runs = [taxonomy]
for i in range(2):
    t, u = generate_taxonomy()
    runs.append(t)
    print(f"Run {i+2}: {t['n_categories']} categories, {u.output_tokens:,} output tokens")

print("\n" + "=" * 72)
print("CATEGORY COUNTS ACROSS RUNS:", [r["n_categories"] for r in runs])
print("=" * 72)

for i, r in enumerate(runs, 1):
    print(f"\nRun {i}:")
    for cat in r["categories"]:
        print(f"  • {cat['name']}")

Run 2: 7 categories, 7,571 output tokens
Run 3: 8 categories, 9,172 output tokens

CATEGORY COUNTS ACROSS RUNS: [8, 7, 8]

Run 1:
  • Terminology & Notation Confusion
  • Faulty Fact or Formula Recall
  • Corrupted Procedure Execution
  • Invalid Rule Generalisation
  • Conceptual Meaning Error
  • Representation Misreading
  • Appearance-Based Spatial Reasoning
  • Task Interpretation & Monitoring

Run 2:
  • Faulty Fact or Definition Recall
  • Conceptual Meaning Gap
  • Overgeneralised or Invented Rule
  • Flawed Procedural Execution
  • Task Misinterpretation
  • Representation Misreading
  • Appearance-Based Assumption

Run 3:
  • Terminology and Notation Confusion
  • Missing or Misremembered Facts
  • Incomplete Conceptual Meaning
  • Invented or Overgeneralised Rule
  • Faulty Procedural Execution
  • Misreading Representations
  • Mistranslating the Question
  • Unwarranted Assumption


**Interpretation.** Consistent category counts and semantically equivalent names
across runs indicate a stable structure. Divergence suggests the corpus does not
cleanly support 6–8 categories, in which case revisit the count constraint before
proceeding to Stage 1b.

Select the run to carry forward in the next cell.

In [25]:
SELECTED_RUN = 0   # 0-indexed; change after reviewing the three runs above

taxonomy = runs[SELECTED_RUN]
category_names = [c["name"] for c in taxonomy["categories"]]

print(f"Selected run {SELECTED_RUN + 1} — {len(category_names)} categories:")
for n in category_names:
    print("  •", n)

Selected run 1 — 8 categories:
  • Terminology & Notation Confusion
  • Faulty Fact or Formula Recall
  • Corrupted Procedure Execution
  • Invalid Rule Generalisation
  • Conceptual Meaning Error
  • Representation Misreading
  • Appearance-Based Spatial Reasoning
  • Task Interpretation & Monitoring


## 8 · Coverage validation

Before committing the taxonomy to 20,520 labeling calls, verify it actually
partitions the corpus it was derived from. Every one of the ~2,587 descriptions
is assigned to a category and the distribution inspected.

Two failure modes this catches:

- **Degenerate distribution** — one category absorbing most of the corpus,
  leaving the classifier with a near-constant target.
- **Unassignable entries** — a meaningful share the model cannot place, which
  means the taxonomy has a genuine gap.

Batched at 100 per call, this is roughly 26 calls on Sonnet.

In [28]:
def response_text(resp):
    """Concatenate text blocks, skipping thinking blocks."""
    parts = [b.text for b in resp.content if getattr(b, "type", None) == "text"]
    if not parts:
        raise ValueError("No text block in response; got: "
                         f"{[getattr(b, 'type', '?') for b in resp.content]}")
    return "".join(parts)

In [29]:
ASSIGN_TEMPLATE = """Assign each numbered misconception to exactly one category.

CATEGORIES:
{cats}

If a misconception genuinely fits none of the categories, assign it "UNASSIGNABLE".
Do not force a poor fit — unassignable entries are diagnostic information.

Return ONLY a JSON object mapping each number to a category name:
{{"1": "<category name>", "2": "<category name>", ...}}

MISCONCEPTIONS:
{items}"""

cats_block = "\n".join(
    f"- {c['name']}: {c['definition']}" for c in taxonomy["categories"]
)


def assign_batch(batch, start_idx, max_retries=3):
    items = "\n".join(f"{start_idx + i + 1}. {d}" for i, d in enumerate(batch))
    last_err = None
    for attempt in range(max_retries):
        try:
            resp = client.messages.create(
                model=LABELING_MODEL,
                max_tokens=16000,   # thinking blocks eat into this budget
                messages=[{"role": "user", "content":
                           ASSIGN_TEMPLATE.format(cats=cats_block, items=items)}],
            )
            return extract_json(response_text(resp))
        except Exception as e:
            last_err = e
            time.sleep(2 ** attempt)
    raise last_err


BATCH_SIZE = 50       # smaller batches leave more room for thinking
assignments = {}
failed_batches = []

for start in range(0, len(descriptions), BATCH_SIZE):
    batch = descriptions[start:start + BATCH_SIZE]
    try:
        assignments.update(assign_batch(batch, start))
    except Exception as e:
        failed_batches.append(start)
        print(f"\n  batch at {start} failed after retries: {e}")
    print(f"\r  assigned {len(assignments):,} / {len(descriptions):,}", end="")

print(f"\n\nTotal assigned : {len(assignments):,} / {len(descriptions):,}")
print(f"Failed batches : {len(failed_batches)}")
if failed_batches:
    print("  offsets:", failed_batches)

  assigned 2,587 / 2,587

Total assigned : 2,587 / 2,587
Failed batches : 0


In [30]:
dist = Counter(assignments.values())
total = sum(dist.values())

print("CATEGORY DISTRIBUTION")
print("=" * 60)
for name, count in dist.most_common():
    bar = "\u2588" * int(40 * count / total)
    print(f"{name[:28]:<28} {count:>5}  {count/total:>5.1%}  {bar}")

unassignable = dist.get("UNASSIGNABLE", 0)
largest = max(v for k, v in dist.items() if k != "UNASSIGNABLE") / total

print("\n" + "=" * 60)
print("HEALTH CHECKS")
print("=" * 60)
print(f"Unassignable share : {unassignable/total:>6.1%}   "
      f"{'PASS' if unassignable/total < 0.05 else 'REVIEW — taxonomy may have a gap'}")
print(f"Largest category   : {largest:>6.1%}   "
      f"{'PASS' if largest < 0.45 else 'REVIEW — distribution is skewed'}")
print(f"Categories in use  : {len([k for k in dist if k != 'UNASSIGNABLE']):>6}   "
      f"of {len(category_names)}")

CATEGORY DISTRIBUTION
Faulty Fact or Formula Recal   565  21.8%  ████████
Conceptual Meaning Error       546  21.1%  ████████
Corrupted Procedure Executio   447  17.3%  ██████
Terminology & Notation Confu   388  15.0%  █████
Invalid Rule Generalisation    256   9.9%  ███
Task Interpretation & Monito   162   6.3%  ██
Representation Misreading      123   4.8%  █
Appearance-Based Spatial Rea   100   3.9%  █

HEALTH CHECKS
Unassignable share :   0.0%   PASS
Largest category   :  21.8%   PASS
Categories in use  :      8   of 8


## 9 · Save the taxonomy artifact

Written to JSON so Stage 1b (notebook 03) can load it directly. Download the file
and commit it to `aai590-capstone/taxonomy/` — the taxonomy is a
project artifact and *should* be version controlled, unlike the raw datasets.

In [31]:
artifact = {
    "generated_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "source_corpus": "Eedi — Mining Misconceptions in Mathematics (Kaggle)",
    "source_n_descriptions": len(descriptions),
    "taxonomy_model": TAXONOMY_MODEL,
    "n_runs_for_stability": len(runs),
    "category_counts_across_runs": [r["n_categories"] for r in runs],
    "selected_run": SELECTED_RUN,
    "taxonomy": taxonomy,
    "validation": {
        "n_assigned": len(assignments),
        "distribution": dict(dist),
        "unassignable_share": unassignable / total,
        "largest_category_share": largest,
    },
}

out = "misconception_taxonomy_v1.json"
with open(out, "w") as f:
    json.dump(artifact, f, indent=2)

print(f"Wrote {out}\n")
print(json.dumps({k: v for k, v in artifact.items() if k != "taxonomy"}, indent=2)[:900])

try:
    from google.colab import files
    files.download(out)
except ImportError:
    pass

Wrote misconception_taxonomy_v1.json

{
  "generated_utc": "2026-07-31T20:22:14Z",
  "source_corpus": "Eedi \u2014 Mining Misconceptions in Mathematics (Kaggle)",
  "source_n_descriptions": 2587,
  "taxonomy_model": "claude-opus-5",
  "n_runs_for_stability": 3,
  "category_counts_across_runs": [
    8,
    7,
    8
  ],
  "selected_run": 0,
  "validation": {
    "n_assigned": 2587,
    "distribution": {
      "Faulty Fact or Formula Recall": 565,
      "Corrupted Procedure Execution": 447,
      "Invalid Rule Generalisation": 256,
      "Representation Misreading": 123,
      "Conceptual Meaning Error": 546,
      "Appearance-Based Spatial Reasoning": 100,
      "Terminology & Notation Confusion": 388,
      "Task Interpretation & Monitoring": 162
    },
    "unassignable_share": 0.0,
    "largest_category_share": 0.21839969076149982
  }
}


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---

## Next steps

1. **Review the taxonomy manually.** The design document specifies human
   validation before use. Read the categories and confirm they are ones a teacher
   would find meaningful and distinct.
2. **Commit the artifact** to `aai590-capstone/taxonomy/misconception_taxonomy_v1.json`.
3. **Stage 1b** (notebook 03) — pilot on ~200 FoundationalASSIST
   (problem, correct answer, wrong answer) triples to measure coverage on the
   actual target data before scaling to all 20,520 pairs.

**Note on the domain shift.** This taxonomy is derived from Eedi's UK-curriculum
diagnostic multiple-choice corpus, while FoundationalASSIST is US Common Core and
predominantly fill-in-the-blank. The Stage 1b pilot exists specifically to measure
whether the categories transfer, and that coverage result belongs in the
Methodology section.

---

*Stage 1a of the two-stage misconception classification pipeline.*